# ATC Speech Enhancement: Training and Evaluation
This notebook implements:
- On-the-fly data augmentation (noise mixing at various SNRs + VHF simulation)
- DeCGAN baseline model (reproduction from the paper)
- Our proposed ATC-EnhanceNet model
- Training and validation loops with PESQ, STOI, SI-SNR evaluation
- Checkpoint saving/loading and prediction on test data

# Key to know

## 0. Annexe

**Generative Adversarial Network (GAN):** is a deep learning framework where two neural networks—a Generator and a Discriminator—compete against each other in a "zero-sum game" to create realistic new data. Introduced by Ian Goodfellow in 2014, GANs are widely used to generate high-quality images, videos, and audio that mimic real-world distributions. The Generator (G) takes random noise as input and tries to transform it into a "fake" data sample (e.g., an image) that looks real. Its goal is to maximize the probability of the Discriminator making a mistake. On the other hand, the Discriminator (D) acts as a classifier that receives both real data from a training set and fake data from the Generator. Its goal is to correctly distinguish between "real" (assigning a score near 1) and "fake" (assigning a score near 0). In our context, the Generator takes noisy speech as input and generates an "enhanced" version, while the Discriminator evaluates pairs of audio samples. It is trained to distinguish between (Clean, Noisy) pairs and (Generated, Noisy) pairs. This "conditional" training ensures the generated speech is not just clean, but also matches the content of the original noisy input.  
[See reference video](https://www.youtube.com/watch?v=-Yxj3yfvY-4)

**Dilated Convolutional:** also known as atrous convolution, is a specialized type of CNN that uses "dilated" filters to expand the receptive field without increasing the number of parameters. By inserting gaps (holes) between kernel elements, these networks can capture a wider global context while maintaining high-resolution spatial information, making them ideal for tasks like image segmentation and audio generation.  
[See reference video](https://www.youtube.com/watch?v=-Yxj3yfvY-4)

**STFT:** Short-Time Fourier Transform is a digital signal processing technique used to analyze how the frequency content of a signal (like speech or music) changes over time. While a standard Fourier Transform (FFT) provides a single snapshot of all frequencies in a signal, the STFT creates a "movie" or spectrogram by dividing the audio into short, overlapping segments.  
[See reference 1](https://blog.prosig.com/2011/05/04/which-should-i-use-real-imaginary-or-magnitude-phase/) | [See reference 2](https://www.youtube.com/watch?v=-Yxj3yfvY-4)

**ISTFT:** Inverse Short-Time Fourier Transform used to convert a processed spectrogram (generated via STFT) back into audible sound.

**Deformable Convolution:** is an advanced version of a standard convolution that can change its "shape" to better fit the objects it is trying to detect.  
[See reference](https://www.sciencedirect.com/topics/computer-science/deformable-convolution#:~:text=6%20Similarly%2C%20in%20instance%20segmentation%2C%20deformable%20convolution,handling%20variations%20in%20size%2C%20shape%2C%20and%20viewpoint)

**Feature‑wise Linear Modulation (FiLM):** a general-purpose conditioning method for neural networks that allows one source of information to influence another. It works by applying a simple, per-feature affine transformation (scaling and shifting) to a network's internal activations based on external conditioning data.  
[See reference](https://www.emergentmind.com/topics/film-layer)

**Complex Ideal Ratio Mask (cIRM):** a time-frequency domain target used in speech enhancement to simultaneously clean both the magnitude and the phase of a noisy signal. Unlike traditional masks like the Ideal Ratio Mask (IRM), which only scale magnitude and ignore phase, the cIRM treats the audio signal as a complex number (Imaginary + Real). By estimating a complex-valued mask, a neural network can theoretically reconstruct the clean speech perfectly from the noisy mixture under additive noise conditions.  
[See reference](https://www.emergentmind.com/topics/complex-ratio-mask-crm)

## 1. DeCGAN Architecture (Baseline)

DeCGAN is a Generative Adversarial Network specifically designed for ATC speech enhancement. Its generator consists of:

- **Encoder**  
  A densely connected dilated convolutional encoder with dilations 1,2,4,8. It takes the concatenated magnitude, real, and imaginary parts of the compressed STFT (shape B×C×T×F, C=3) and outputs a feature map with compressed frequency dimension (F/2). The dense connections aggregate multi‑scale information.

- **DeConformer Blocks (5 blocks)**  
  Each block contains two sub‑modules:

  - **TFC‑SA (Time‑Frequency‑Channel Self‑Attention)**  
    Computes attention independently along the channel, time, and frequency dimensions using global pooling and 1×1 convolutions. It then combines the three attention maps to modulate the input, capturing global dependencies across all dimensions.

  - **DeConv‑FFN (Deformable Convolution Feed‑Forward Network)**  
    Two parallel branches with different numbers of deformable convolution layers (2 and 4) to capture local features at different receptive fields. Their outputs are fused via channel attention, allowing flexible local feature extraction.

- **Dual Decoders**  
  - **Mask Decoder:** Predicts a magnitude mask (values in [0,1]) from the DeConformer output. This mask is applied to the noisy magnitude spectrum.  
  - **Complex Decoder:** Predicts a 2‑channel (real/imag) refinement that is added to the original noisy complex spectrum (or used to reconstruct phase). The two decoder outputs are combined to form the enhanced complex spectrogram.

- **Metric Discriminator**  
  A convolutional network that takes magnitude spectrograms and outputs a scalar quality score. It is trained to estimate the PESQ score of the enhanced signal, guiding the generator to produce perceptually high‑quality output.

**Losses:**  
- Combined TF loss (L_TF) = weighted sum of magnitude L2 and real/imag L2.  
- Adversarial loss (L_GAN) (least‑squares).  
- Time‑domain L1 loss (L_Time).

## 2. ATC-EnhanceNet (Our Proposed Model)

Our model retains the DeConformer blocks but introduces several modifications aimed at extreme‑SNR conditions and real‑time deployment.

- **Encoder (same as DeCGAN)**  
  Still uses dense dilated convolutions to extract multi‑scale features and compress frequency.

- **SNR Estimator**  
  A lightweight auxiliary network that performs global average pooling on the encoder output, followed by two linear layers, to produce a scalar embedding that represents the estimated SNR of the input. This embedding is later injected into the decoder to adapt the enhancement based on noise level.

- **Multi‑Scale Temporal Convolution**  
  A stack of dilated 1D convolutions (WaveNet‑style) applied on the time dimension after the encoder. It expands the temporal receptive field, enabling the model to capture long‑range dependencies (e.g., across words or syllables) without increasing the number of DeConformer blocks.

- **DeConformer Blocks (same as DeCGAN)**  
  Used for joint time‑frequency‑channel modelling.

- **FiLM Modulation**  
  Instead of using dual decoders, we apply Feature‑wise Linear Modulation (FiLM) to the DeConformer output. The SNR embedding is used to compute per‑channel scale (γ) and shift (β) parameters, which are applied to the feature maps. This allows the model to dynamically adjust its behaviour according to the estimated noise level.

- **Unified Complex Mask Decoder**  
  A single convolutional decoder that outputs two channels (real and imaginary) representing a complex ideal ratio mask (cIRM). The mask is upsampled back to the original frequency dimension and applied directly to the noisy complex STFT. This avoids the complexity of merging two separate decoder outputs and simplifies the loss.

**Losses (same as DeCGAN plus perceptual loss):**  
- Multi‑resolution STFT loss (L1 across different window sizes)  
- Complex magnitude loss (L2 on magnitude)  
- Perceptual loss (feature matching using a pre‑trained ATC‑ASR model)  
- Adversarial loss (with metric discriminator)  
- Time‑domain L1 loss

## Targeted Performance Metrics

To evaluate our speech enhancement model, we use four complementary metrics that capture different aspects of quality and utility.

| Metric | Abbreviation | Description | Range | Target | Relevance |
|--------|--------------|-------------|-------|--------|-----------|
| Perceptual Evaluation of Speech Quality | **PESQ** | Standardized objective measure of speech quality, predicting mean opinion score (MOS) under clean and noisy conditions. Higher values indicate better quality. | -0.5 to 4.5 | **> 3.40** | Ensures the enhanced speech sounds natural and pleasant to human listeners. |
| Short‑Time Objective Intelligibility | **STOI** | Predicts the intelligibility of speech in noisy environments. Values closer to 1 indicate perfect understanding. | 0 to 1 | **> 0.97** | Guarantees that the enhanced speech remains understandable, critical for safety‑critical ATC communications. |
| Scale‑Invariant Signal‑to‑Noise Ratio | **SI‑SNR** | Measures the ratio of the clean signal to the residual distortion after enhancement, ignoring scaling factors. Higher values mean less distortion. | -∞ to ∞ (dB) | **> 12 dB** | Quantifies the amount of noise suppression while preserving speech content. |
| Word Error Rate | **WER** (ASR) | Extrinsic metric: error rate of an automatic speech recognition system on the enhanced audio. Lower is better. | 0 to 1 (or %) | **≥15% absolute reduction** | Demonstrates real‑world utility: improving downstream tasks like ASR, which is essential for ATC automation. |

Our primary goal is to surpass the current ATC‑specific state‑of‑the‑art (DeCGAN, PESQ 3.31 / STOI 0.96) while achieving a meaningful reduction in ASR Word Error Rate (≥15% absolute) and real‑time operation (RTF < 0.1).

## Setup

### PSC only

#### **1. Navigate to Your Directory**
Make sure to use this `/ocean/projects/cis260061p/<your_username>` as your working directory.
```bash
cd /ocean/projects/cis260061p/<your_username>  #change <your_username> to your username
```

In [ ]:
!cd /ocean/projects/cis260061p/<psc username>
!pwd

#### **2. Request a Compute Node**
```bash
interact -p GPU-shared --gres=gpu:v100-32:1 -t 8:00:00 -A cis260061p
```

#### **3. Activate the provided HW3 Environment**
```bash
conda deactivate # First, deactivate any existing Conda environment
conda activate /ocean/projects/cis260061p/shared/envs/atc-enhance
```

#### 4. Configure Kaggle API Access

⚠️ **Important:** You must provide your own Kaggle API credentials for data downloads, while creating your deliverables for code submission and (optionally) submitting to the competition.

#### How to obtain your Kaggle API credentials

1. Navigate to your **Kaggle Profile → Settings** tab.
2. Click **“Genetate New Token”**
3. Copy and paste the access token below


In [ ]:
import os
import pprint
os.environ['KAGGLE_USERNAME'] = "<username>" # TODO: Verify in Settings
os.environ['KAGGLE_API_TOKEN'] = "<api token>" # TODO: Add Access Token (must be the new token starting with "KGAT_")

# Verify
import kaggle
api = kaggle.api  # Already authenticated on import

#### 5. Get Data

In [ ]:
!mkdir -p $LOCAL/dataset

# ATCO2-ASRdataset-v1_beta
api.dataset_download_files('atco2-asrdataset-v1-beta', path=os.environ["LOCAL"])
!unzip -qo $LOCAL/atco2-asrdataset-v1-beta.zip -d $LOCAL/dataset
!rm -f $LOCAL/atco2-asrdataset-v1-beta.zip

# VoiceBank-transcripts-testset
api.dataset_download_files('voicebank-transcripts-testset', path=os.environ["LOCAL"])
!unzip -qo $LOCAL/voicebank-transcripts-testset.zip -d $LOCAL/dataset
!rm -f $LOCAL/voicebank-transcripts-testset.zip

# VoiceBank_DEMAND_16k
api.dataset_download_files('voicebank-demand-16k', path=os.environ["LOCAL"])
!unzip -qo $LOCAL/voicebank-demand-16k.zip -d $LOCAL/dataset
!rm -f $LOCAL/voicebank-demand-16k.zip

# WHAM noise reduced dataset
api.dataset_download_files('wham-noise-reduced-dataset', path=os.environ["LOCAL"])
!unzip -qo $LOCAL/wham-noise-reduced-dataset.zip -d $LOCAL/dataset
!rm -f $LOCAL/wham-noise-reduced-dataset.zip

###  Kaggle

#### Install packages

In [ ]:
!pip install librosa soundfile numpy scipy matplotlib tqdm pandas pesq pystoi jiwer transformers --quiet

In [ ]:
import os
import pprint
os.environ['KAGGLE_USERNAME'] = "<username>" # TODO: Verify in Settings
os.environ['KAGGLE_API_TOKEN'] = "<api token>" # TODO: Add Access Token (must be the new token starting with "KGAT_")

# Verify
import kaggle
api = kaggle.api  # Already authenticated on import

In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import librosa
import soundfile as sf
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import matplotlib.pyplot as plt
import scipy.signal
from pesq import pesq
from pystoi import stoi
from jiwer import wer
from transformers import Wav2Vec2Processor, Wav2Vec2Model
import warnings
warnings.filterwarnings('ignore')

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Constants
SAMPLE_RATE = 16000
N_FFT = 512
HOP_LENGTH = 256
WIN_LENGTH = 512
POWER_LAW_C = 0.3
SEGMENT_DURATION = 2  # seconds (for splitting)

### Check datasets

  - `Kaggle:`: `"/kaggle/input/datasets/<username>"`
  - `PSC`: `"/local/dataset"`

In [ ]:
data_path = "/kaggle/input/datasets/<username>" # this is the path of the dataset
print("Files in shared dataset:", os.listdir(data_path))

## 1. Config

In [ ]:
%%writefile config.yaml

batch_size: 16 
epochs: 100

# Optimizer lr for Generator and Discriminator
optimizer_G_lr: 0.0005
optimizer_D_lr: 0.001

# gradient clipping
gradient_clipping: 10.0

# Losses hyperparameters
mag_ri_weight: 0.7 # weight to combine magnitude loss LMag and the phase-aware loss LRI, 0.7 in the baseline paper DecGAN
lambda_Ltf: 1 # combined loss Ltf weight, 1 in the baseline paper DecGAN
lambda_Lgan: 0.05 # adversarial loss Lgan weight, 0.05 in the baseline paper DecGAN
lambda_Lmrstft: 0.2 # Multi-Resolution STFT Loss weight, replace the time-domain loss in the baseline paper DecGAN which has 0.2 weight in the total loss
lambda_Lpercep: 0.1 # new added perceptual loss Lpercep weight, 0.1 default

# Others
wandb: True
Name: "Anderson Awolossou" # Write your name here

In [ ]:
import yaml
with open("config.yaml") as file:
    config = yaml.safe_load(file)

config

## 2. Data Utilities
We define a dataset class that loads clean segments (or full files) and generates noisy versions on the fly by mixing with random noise at a random SNR, optionally applying VHF simulation. The dataset returns `(noisy_waveform, clean_waveform, snr)`.

In [ ]:
def vhf_simulation(audio, sr, prob_distort=0.3, prob_fade=0.2):
    """Apply VHF-like distortions (bandpass, optional clipping, fading)."""
    # Bandpass filter (300-3400 Hz)
    sos = scipy.signal.butter(10, [300, 3400], btype='band', fs=sr, output='sos')
    audio = scipy.signal.sosfilt(sos, audio)
    
    # Random non-linear distortion
    if np.random.rand() < prob_distort:
        threshold = 0.8 * np.max(np.abs(audio))
        audio = np.tanh(audio / threshold) * threshold
    
    # Random fading
    if np.random.rand() < prob_fade:
        t = np.arange(len(audio)) / sr
        depth = 0.3
        rate = 5.0
        mod = 1 - depth * (0.5 + 0.5 * np.sin(2 * np.pi * rate * t))
        audio = audio * mod
    return audio

def add_noise(clean, noise, snr_db):
    """Mix clean and noise at given SNR."""
    clean_rms = np.sqrt(np.mean(clean**2))
    noise_rms = np.sqrt(np.mean(noise**2))
    desired_noise_rms = clean_rms / (10**(snr_db/20))
    scale = desired_noise_rms / (noise_rms + 1e-10)
    noisy = clean + noise * scale
    # Normalize to avoid clipping
    max_val = np.max(np.abs(noisy))
    if max_val > 0.95:
        noisy = noisy / max_val * 0.95
    return noisy

Datasets locations :
  - `Kaggle`: `"<root>/atco2-asrdataset-v1-beta/ATCO2-ASRdataset-v1_beta/DATA"`, `"<root>/voicebank-demand-16k/noisy_trainset_28spk_wav/"`, `"<root>/wham-noise-reduced-dataset/wham_noise/"`
  - `PSC`: `"<root>/ATCO2-ASRdataset-v1_beta/DATA"`, `"<root>/noisy_trainset_28spk_wav/"`, `"<root>/wham_noise/"`

In [ ]:
import glob
import pandas as pd

data_root = data_path 

# Clean
atc_root = f"{data_root}/atco2-asrdataset-v1-beta/ATCO2-ASRdataset-v1_beta/DATA" # change if PSC
clean_files = sorted(glob.glob(os.path.join(atc_root, "*.wav")))
xml_files = sorted(glob.glob(os.path.join(atc_root, "*.xml")))

print(f"Found {len(clean_files)} audio files and {len(xml_files)} XML files.")
# Create a DataFrame with file info
file_info = []
for wav_path in clean_files:
    # Corresponding XML: same basename but .xml
    base = os.path.splitext(os.path.basename(wav_path))[0]
    xml_path = os.path.join(atc_root, base + ".xml")
    #print(xml_path, wav_path)
    if os.path.exists(xml_path):
        # Get duration using librosa (load metadata only)
        dur = librosa.get_duration(filename=wav_path)
        file_info.append({
            'filename': base,
            'wav_path': wav_path,
            'xml_path': xml_path,
            'duration': dur
        })

df_files = pd.DataFrame(file_info)
print(f"Total usable files: {len(df_files)}")

# Plot duration distribution
plt.figure(figsize=(8,4))
plt.hist(df_files['duration'], bins=30, edgecolor='black')
plt.xlabel('Duration (seconds)')
plt.ylabel('Count')
plt.title('Distribution of Audio File Durations')
plt.grid(alpha=0.3)
plt.savefig("atc_audio_durations.png", dpi=300)
plt.show()

print(f"Average duration: {df_files['duration'].mean():.2f}s")
print(f"Total audio: {df_files['duration'].sum() / 3600:.2f} hours")

In [ ]:
# Noisy
vb_demand_path = f"{data_root}/voicebank-demand-16k/noisy_trainset_28spk_wav/"  # change if PSC
whamvox_path = f"{data_root}/wham-noise-reduced-dataset/wham_noise/"            # change if PSC

# Collect all noise files
noise_files = glob.glob(os.path.join(vb_demand_path, "*.wav")) + \
              glob.glob(os.path.join(whamvox_path, "*.wav"))
print(f"Found {len(noise_files)} noise files.")

target_snrs = [-20, -15, -10, -5, 0, 5] # different levels of noise targeted

In [ ]:
class ATCAugmentedDataset(Dataset):
    """Loads clean segments (or full files) and generates noisy variants on the fly.
    
    Args:
        clean_files (list): paths to clean audio files (or segments)
        noise_files (list): paths to noise files
        target_snrs (list): list of SNR values in dB, e.g., [-20, -15, -10, -5, 0, 5]
        prob_vhf (float): probability of applying VHF simulation to the mixed signal
        segment_duration (float): if not None, each file is split into fixed-length segments
        val (boolean): if validation data, default is false
    """
    def __init__(self, clean_files, noise_files, target_snrs, prob_vhf=1.0, segment_duration=None, val=False):
        self.clean_files = clean_files
        self.noise_files = noise_files
        self.target_snrs = target_snrs
        self.prob_vhf = prob_vhf
        self.segment_duration = segment_duration

        # Split the dataset, first 0.5 for training and last 0.05 for validation
        N = len(clean_files)

        train_end = int(0.5 * N)
        val_start  = int(0.95 * N)
        
        train_files = clean_files[:train_end]
        val_files   = clean_files[val_start:]

        # Select subset of data to use for train or val
        self.clean_files = val_files if val else train_files
        
        # Pre-load all clean audio into memory? Not necessary; we'll load on demand.
        # But we need to handle splitting into segments.
        if segment_duration is not None:
            self.segments = []
            for f in self.clean_files:
                audio, sr = librosa.load(f, sr=SAMPLE_RATE, mono=True)
                # split into non-overlapping segments of segment_duration seconds
                n_samples = int(segment_duration * SAMPLE_RATE)
                for start in range(0, len(audio), n_samples):
                    seg = audio[start:start+n_samples]
                    if len(seg) == n_samples:
                        self.segments.append(seg)
            print(f"Created {len(self.segments)} segments from {len(clean_files)} files")
        else:
            # Keep full files, will load each time
            self.segments = None
    
    def __len__(self):
        if self.segments is not None:
            return len(self.segments) * len(self.target_snrs)  # each segment paired with each SNR? Actually we will sample randomly.
        else:
            return len(self.clean_files) * len(self.target_snrs)  # but we'll sample on the fly.
    
    def __getitem__(self, idx):
        # Instead of precomputing all combinations, we sample randomly each time.
        # This is more efficient for on-the-fly augmentation.
        # We'll ignore idx and just return a random sample.
        # Choose random clean segment/file
        if self.segments is not None:
            clean = random.choice(self.segments)
        else:
            clean_file = random.choice(self.clean_files)
            clean, _ = librosa.load(clean_file, sr=SAMPLE_RATE, mono=True)
        
        # Random SNR
        snr = random.choice(self.target_snrs)
        
        # Random noise file
        noise_file = random.choice(self.noise_files)
        noise, _ = librosa.load(noise_file, sr=SAMPLE_RATE, mono=True)
        
        # Ensure noise length matches clean
        if len(noise) < len(clean):
            repeats = int(np.ceil(len(clean) / len(noise)))
            noise = np.tile(noise, repeats)
        noise = noise[:len(clean)]
        
        # Add noise
        noisy = add_noise(clean, noise, snr)
        
        # Optionally apply VHF simulation
        if np.random.rand() < self.prob_vhf:
            noisy = vhf_simulation(noisy, SAMPLE_RATE)

        # Normalization
        scale = max(np.max(np.abs(clean)), np.max(np.abs(noisy))) + 1e-8
        clean = clean / scale
        noisy = noisy / scale
        
        # Convert to torch tensors
        #clean_t = torch.from_numpy(clean).float().unsqueeze(0)  # (1, T)
        #noisy_t = torch.from_numpy(noisy).float().unsqueeze(0)
        clean_t = torch.from_numpy(clean).float()
        noisy_t = torch.from_numpy(noisy).float()
        
        
        return noisy_t, clean_t, snr

In [ ]:
class ATCAugmentedDatasetTest(Dataset):
    """Loads clean segments (or full files) and generates noisy variants on the fly.
    
    Args:
        clean_files (list): paths to clean audio files (or segments)
        segment_duration (float): if not None, each file is split into fixed-length segments
    """
    def __init__(self, clean_files, xml_files, data_root=atc_root):
        self.clean_files = clean_files
        
        # Split the dataset, first 0.5 for training and last 0.1 for validation, last 0.5 for TEST
        N = len(clean_files)

        train_end = int(0.5 * N)
        
        #train_files = clean_files[:train_end]
        test_files  = clean_files #[train_end:]
        #val_files   = clean_files[test_end:]

        # Select subset of data to use for train or val
        self.clean_files = []
        self.xml_s = []
        
        for f in test_files:
            # parse xml
            base = os.path.splitext(os.path.basename(f))[0]
            xml_path = os.path.join(data_root, base + ".xml")
            if os.path.exists(xml_path): # Proceed if only xml transcripts exist
                
                self.clean_files.append(f)
                self.xml_s.append(xml_path)
                
        print(f"Extracted {len(self.clean_files)}, {len(self.xml_s)} (audio,xml) from {len(test_files)} files")
    
    def __len__(self):
        return len(self.clean_files)
    
    def __getitem__(self, idx):
        clean_file = self.clean_files[idx]        
        xml_file = self.xml_s[idx]
        
        clean, _ = librosa.load(clean_file, sr=SAMPLE_RATE, mono=True)
        clean_t = torch.from_numpy(clean).float()  # (T,)
        
        return clean_t, xml_file  # return both directly

    def collate_fn(self, batch):
        """
        batch: list of tuples (clean_tensor, xml_file)
        """
        clean_tensors = [item[0] for item in batch]
        xml_files = [item[1] for item in batch]
        
        # Optionally pad sequences to same length for batch
        clean_tensors = torch.nn.utils.rnn.pad_sequence(clean_tensors, batch_first=True)
        
        return clean_tensors, xml_files

### 2.1 Data loaders

In [ ]:
# To free up ram
import gc
gc.collect()

In [ ]:
# Create a dataset object using the AudioDataset class for the training data
train_data = ATCAugmentedDataset(clean_files, noise_files, target_snrs, prob_vhf=1.0, segment_duration=2.0)

# Create a dataset object using the AudioDataset class for the validation data
val_data = ATCAugmentedDataset(clean_files, noise_files, target_snrs, prob_vhf=1.0, segment_duration=2.0, val=True)

# Create a dataset object using the AudioTestDataset class for the test data
test_data = ATCAugmentedDatasetTest(clean_files, xml_files)

In [ ]:

# dataloader = DataLoader(dataset, batch_size=16, shuffle=True, num_workers=4)

train_loader = torch.utils.data.DataLoader(
    dataset     = train_data,
    num_workers = 4,
    batch_size  = config['batch_size'],
    pin_memory  = True,
    shuffle     = True,
    #collate_fn = train_data.collate_fn
)

val_loader = torch.utils.data.DataLoader(
    dataset     = val_data,
    num_workers = 0,
    batch_size  = config['batch_size'],
    pin_memory  = True,
    shuffle     = False,
    #collate_fn = val_data.collate_fn
)

test_loader = torch.utils.data.DataLoader(
    dataset     = test_data,
    num_workers = 0,
    batch_size  = config['batch_size'],
    pin_memory  = True,
    shuffle     = False,
    collate_fn = test_data.collate_fn
)


print("Batch size: ", config['batch_size'])
print("Train dataset samples = {}, batches = {}".format(train_data.__len__(), len(train_loader)))
print("Val dataset samples = {}, batches = {}".format(val_data.__len__(), len(val_loader)))
print("Test dataset samples = {}, batches = {}".format(test_data.__len__(), len(test_loader)))

In [ ]:
# sanity check
for data in train_loader:
    x, y, snr = data
    print(x.shape, y.shape, snr.shape)
    break

## 3. STFT and Complex Feature Extraction
We define a module that converts a waveform to a complex spectrogram, applies power-law compression, and returns the concatenated magnitude, real, and imaginary parts as a tensor of shape `(B, T, F, 3)`.

In [ ]:
class STFTModule(nn.Module):
    def __init__(self, n_fft=N_FFT, hop_length=HOP_LENGTH, win_length=WIN_LENGTH, power_law_c=POWER_LAW_C):
        super().__init__()
        self.n_fft = n_fft
        self.hop_length = hop_length
        self.win_length = win_length
        self.c = power_law_c
        self.register_buffer('window', torch.hann_window(win_length))
    
    def forward(self, waveform):
        # waveform: (B, T)
        spec = torch.stft(waveform, n_fft=self.n_fft, hop_length=self.hop_length,
                          win_length=self.win_length, window=self.window,
                          return_complex=True)  # (B, F, T)
        mag = torch.abs(spec)  # (B, F, T)
        phase = torch.angle(spec)
        mag_compressed = mag ** self.c
        real = mag_compressed * torch.cos(phase)
        imag = mag_compressed * torch.sin(phase)
        # Permute to (B, T, F, 3)
        out = torch.stack([mag_compressed, real, imag], dim=-1)  # (B, F, T, 3)
        out = out.permute(0, 2, 1, 3)  # (B, T, F, 3)
        return out, spec  # spec is full complex for later use

    def inverse(self, mag_compressed, real, imag, original_length=None):
        phase = torch.atan2(imag, real + 1e-10)
        mag = mag_compressed ** (1/self.c)
        spec = mag * torch.exp(1j * phase)
        waveform = torch.istft(spec, n_fft=self.n_fft, hop_length=self.hop_length,
                               win_length=self.win_length, window=self.window,
                               length=original_length)
        return waveform

## 4. Model Definitions
We implement:
- The DeCGAN generator and discriminator (as described in the paper).
- Our proposed ATC-EnhanceNet (unified complex mask decoder + SNR conditioning).
Note: Some components are simplified for clarity (e.g., using regular convolutions instead of deformable).

In [ ]:
class DenseDilatedConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, dilation):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=3,
                              padding=dilation, dilation=dilation)
        self.relu = nn.LeakyReLU(0.2)

    def forward(self, x):
        return self.relu(self.conv(x))

class Encoder(nn.Module):
    def __init__(self, in_channels=3, base_channels=64):
        super().__init__()
        self.conv_in = nn.Conv2d(in_channels, base_channels, kernel_size=3, padding=1)
        self.block1 = DenseDilatedConvBlock(base_channels, base_channels, dilation=1)
        self.block2 = DenseDilatedConvBlock(base_channels*2, base_channels, dilation=2)
        self.block3 = DenseDilatedConvBlock(base_channels*3, base_channels, dilation=4)
        self.block4 = DenseDilatedConvBlock(base_channels*4, base_channels, dilation=8)
        self.compress_freq = nn.Conv2d(base_channels*5, base_channels, kernel_size=(3,1),
                                       stride=(2,1), padding=(1,0))
        self.relu = nn.LeakyReLU(0.2)

    def forward(self, x):
        # x: (B, C, T, F) with C=3 (mag, real, imag)
        x = self.relu(self.conv_in(x))
        d1 = self.block1(x)
        d2 = self.block2(torch.cat([x, d1], dim=1))
        d3 = self.block3(torch.cat([x, d1, d2], dim=1))
        d4 = self.block4(torch.cat([x, d1, d2, d3], dim=1))
        out = torch.cat([x, d1, d2, d3, d4], dim=1)
        out = self.compress_freq(out)
        return out  # (B, C, T, F/2)

class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.gap(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y

class TFC_SA(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1x1 = nn.Conv2d(channels, channels, kernel_size=1)
        self.channel_att = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, channels, kernel_size=1),
            nn.Sigmoid()
        )
        self.time_att = nn.Sequential(
            nn.AdaptiveAvgPool2d((None, 1)),
            nn.Conv2d(channels, channels, kernel_size=1),
            nn.Sigmoid()
        )
        self.freq_att = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, None)),
            nn.Conv2d(channels, channels, kernel_size=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        att_c = self.channel_att(x)
        att_t = self.time_att(x)
        att_f = self.freq_att(x)
        att = att_c * att_t * att_f
        out = self.conv1x1(x * att)
        return out

class DeConvFFN(nn.Module):
    def __init__(self, channels):
        super().__init__()
        # Simplified version using regular conv with different kernel sizes
        self.branch1 = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=3, padding=1),
            nn.LeakyReLU(0.2),
            nn.Conv2d(channels, channels, kernel_size=3, padding=1),
        )
        self.branch2 = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=5, padding=2),
            nn.LeakyReLU(0.2),
            nn.Conv2d(channels, channels, kernel_size=5, padding=2),
        )
        self.ca = ChannelAttention(channels*2)
        self.fusion = nn.Conv2d(channels*2, channels, kernel_size=1)

    def forward(self, x):
        b1 = self.branch1(x)
        b2 = self.branch2(x)
        concat = torch.cat([b1, b2], dim=1)
        concat = self.ca(concat)
        out = self.fusion(concat)
        return out

class DeConformerBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.norm1 = nn.LayerNorm([channels])
        self.tfc_sa = TFC_SA(channels)
        self.norm2 = nn.LayerNorm([channels])
        self.deconv_ffn = DeConvFFN(channels)

    def forward(self, x):
        # x: (B, C, T, F) where C = channels
        # LayerNorm expects (B, T, F, C) shape, so permute
        B, C, T, F = x.shape
        x_perm = x.permute(0, 2, 3, 1)  # (B, T, F, C)
        x_perm = self.norm1(x_perm)
        x_perm = x_perm.permute(0, 3, 1, 2)  # back to (B, C, T, F)
        attn = self.tfc_sa(x_perm)
        x = x + attn
        # FFN part
        x_perm = x.permute(0, 2, 3, 1)
        x_perm = self.norm2(x_perm)
        x_perm = x_perm.permute(0, 3, 1, 2)
        ffn = self.deconv_ffn(x_perm)
        x = x + ffn
        return x

class MaskDecoder(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1),
            nn.LeakyReLU(0.2),
            nn.Conv2d(in_channels, 1, kernel_size=3, padding=1),
            nn.Sigmoid()
        )
        self.upsample = nn.Upsample(scale_factor=(2, 1), mode='bilinear', align_corners=False)

    def forward(self, x):
        # x: (B, C, T, F/2)
        mask = self.conv(x)  # (B, 1, T, F/2)
        mask = self.upsample(mask)  # (B, 1, T, F)
        return mask

class ComplexDecoder(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1),
            nn.LeakyReLU(0.2),
            nn.Conv2d(in_channels, 2, kernel_size=3, padding=1),
            nn.Tanh()
        )
        self.upsample = nn.Upsample(scale_factor=(2, 1), mode='bilinear', align_corners=False)

    def forward(self, x):
        comp = self.conv(x)  # (B, 2, T, F/2)
        comp = self.upsample(comp)  # (B, 2, T, F)
        return comp  # real, imag

class MetricDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_blocks = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, stride=2, padding=1),
            nn.InstanceNorm2d(16),
            nn.PReLU(),
            nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1),
            nn.InstanceNorm2d(32),
            nn.PReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.InstanceNorm2d(64),
            nn.PReLU(),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.InstanceNorm2d(128),
            nn.PReLU(),
        )
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(128, 64),
            nn.PReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, mag):
        # mag: (B, 1, T, F)
        x = self.conv_blocks(mag)
        x = self.gap(x).view(x.size(0), -1)
        score = self.fc(x)
        return score

class DeCGANGenerator(nn.Module):
    def __init__(self, in_channels=3, base_channels=64, num_blocks=5):
        super().__init__()
        self.encoder = Encoder(in_channels, base_channels)
        self.deconformer_blocks = nn.ModuleList([DeConformerBlock(base_channels) for _ in range(num_blocks)])
        self.mask_decoder = MaskDecoder(base_channels)
        self.complex_decoder = ComplexDecoder(base_channels)

    def forward(self, x):
        # x: (B, C, T, F) with C=3
        enc = self.encoder(x)  # (B, base_channels, T, F/2)
        for block in self.deconformer_blocks:
            enc = block(enc)
        mask = self.mask_decoder(enc)   # (B, 1, T, F)
        comp = self.complex_decoder(enc) # (B, 2, T, F)
        # Combine: mask for magnitude, comp for real/imag refinement? The paper merges outputs.
        # For simplicity, we apply mask to the magnitude of the input and use comp to replace real/imag.
        # But this is an approximation. We'll return both and let the loss handle them.
        return mask, comp

In [ ]:
class SNRFiLMLayer(nn.Module):
    def __init__(self, channels, snr_embed_dim=1):
        super().__init__()
        self.gamma = nn.Linear(snr_embed_dim, channels)
        self.beta = nn.Linear(snr_embed_dim, channels)

    def forward(self, x, snr_emb):
        # x: (B, C, T, F), snr_emb: (B, embed_dim)
        gamma = self.gamma(snr_emb).unsqueeze(-1).unsqueeze(-1)  # (B, C, 1, 1)
        beta = self.beta(snr_emb).unsqueeze(-1).unsqueeze(-1)
        return x * gamma + beta

class SNREstimator(nn.Module):
    def __init__(self, in_channels, embed_dim=1):
        super().__init__()
        # Lightweight network: global pooling + MLP
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(in_channels, 32),
            nn.ReLU(),
            nn.Linear(32, embed_dim)
        )

    def forward(self, x):
        # x: (B, C, T, F)
        x = self.gap(x).squeeze(-1).squeeze(-1)  # (B, C)
        emb = self.fc(x)  # (B, embed_dim)
        return emb

class ATCEnhanceNetGenerator(nn.Module):
    def __init__(self, in_channels=3, base_channels=64, num_blocks=5, snr_embed_dim=1):
        super().__init__()
        self.encoder = Encoder(in_channels, base_channels)
        self.snr_estimator = SNREstimator(base_channels, snr_embed_dim)
        self.temporal_conv = nn.Sequential(
            nn.Conv2d(base_channels, base_channels, kernel_size=(1, 1)),
            nn.LeakyReLU(0.2),
            # WaveNet-style dilated convs on time dimension
            nn.Conv2d(base_channels, base_channels, kernel_size=(3, 1), padding=(2, 0), dilation=(2, 1)),
            nn.LeakyReLU(0.2),
            nn.Conv2d(base_channels, base_channels, kernel_size=(3, 1), padding=(4, 0), dilation=(4, 1)),
            nn.LeakyReLU(0.2),
        )
        self.deconformer_blocks = nn.ModuleList([DeConformerBlock(base_channels) for _ in range(num_blocks)])
        self.film = SNRFiLMLayer(base_channels, snr_embed_dim)
        # Unified complex mask decoder (outputs 2 channels: real and imag)
        self.complex_decoder = nn.Sequential(
            nn.Conv2d(base_channels, base_channels, kernel_size=3, padding=1),
            nn.LeakyReLU(0.2),
            nn.Conv2d(base_channels, 2, kernel_size=3, padding=1),
            nn.Tanh()
        )
        self.upsample = nn.Upsample(scale_factor=(2, 1), mode='bilinear', align_corners=False)

    def forward(self, x):
        # x: (B, C, T, F) with C=3
        enc = self.encoder(x)  # (B, base_channels, T, F/2)
        snr_emb = self.snr_estimator(enc)  # (B, embed_dim)
        # Temporal convolution
        enc = self.temporal_conv(enc)
        for block in self.deconformer_blocks:
            enc = block(enc)
        # FiLM conditioning
        enc = self.film(enc, snr_emb)
        # Decode to complex mask (2 channels: real, imag)
        comp = self.complex_decoder(enc)  # (B, 2, T, F/2)
        comp = self.upsample(comp)  # (B, 2, T, F)
        # The mask is now directly the complex mask (cIRM). We'll apply it to the noisy STFT.
        return comp, snr_emb  # return mask (real/imag) and SNR embedding for monitoring

In [ ]:
stft_module = STFTModule().to(device)
# Choose which model to train
generator = ATCEnhanceNetGenerator().to(device)  # or DeCGANGenerator()
discriminator = MetricDiscriminator().to(device)

In [ ]:
from torchinfo import summary

noisy_stft, noisy_complex = stft_module(x.to(device))
clean_stft, clean_complex = stft_module(y.to(device))
# For generator input: concatenated magnitude, real, imag (shape: B, T, F, 3) -> permute to (B,3,T,F)
gen_input = noisy_stft.permute(0,3,1,2)  # (B,3,T,F)
        
summary(generator, input_data=gen_input)

## 5. Training Utilities
Define loss functions (MRSTFT, complex magnitude, perceptual, adversarial), training loop, validation, and checkpointing.

In [ ]:
class MultiResolutionSTFTLoss(nn.Module):
    def __init__(self, fft_sizes=[256, 512, 1024], hop_sizes=[64, 128, 256], win_lengths=[256, 512, 1024], window='hann'):
        super().__init__()
        self.fft_sizes = fft_sizes
        self.hop_sizes = hop_sizes
        self.win_lengths = win_lengths
        self.window = window

    def forward(self, x, y):
        # x, y: waveforms (B, T)
        loss = 0
        for fft, hop, win in zip(self.fft_sizes, self.hop_sizes, self.win_lengths):
            X = torch.stft(x, n_fft=fft, hop_length=hop, win_length=win,
                           window=torch.hann_window(win).to(x.device), return_complex=True)
            Y = torch.stft(y, n_fft=fft, hop_length=hop, win_length=win,
                           window=torch.hann_window(win).to(y.device), return_complex=True)
            loss += F.l1_loss(X.abs(), Y.abs())
        return loss / len(self.fft_sizes)

class ComplexMagnitudeLoss(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, mag_pred, mag_target):
        return F.l1_loss(mag_pred, mag_target)

class PerceptualLoss(nn.Module):
    def __init__(self, model_name="facebook/wav2vec2-base-960h"):
        super().__init__()
        self.processor = Wav2Vec2Processor.from_pretrained(model_name)
        self.model = Wav2Vec2Model.from_pretrained(model_name)
        for param in self.model.parameters():
            param.requires_grad = False
        self.model.to(device)

    def forward(self, clean, enhanced):
        # clean, enhanced: waveforms (B, T)
        # We need to pass through processor and model, extract features from intermediate layers
        # For simplicity, we'll use the last hidden state    
        clean = clean.detach().cpu().numpy()
        enhanced = enhanced.detach().cpu().numpy()
    
        inputs = self.processor(clean, sampling_rate=16000, return_tensors="pt", padding=True)
        inputs_enh = self.processor(enhanced, sampling_rate=16000, return_tensors="pt", padding=True)

        with torch.no_grad():
            clean_feat = self.model(inputs.input_values.to(device)).last_hidden_state
            enhanced_feat = self.model(inputs_enh.input_values.to(device)).last_hidden_state
        # Pad to same length if needed
        min_len = min(clean_feat.size(1), enhanced_feat.size(1))
        clean_feat = clean_feat[:, :min_len, :]
        enhanced_feat = enhanced_feat[:, :min_len, :]
        return F.mse_loss(clean_feat, enhanced_feat)

class AdversarialLoss(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, disc_output, target_real=True):
        if target_real:
            return F.mse_loss(disc_output, torch.ones_like(disc_output))
        else:
            return F.mse_loss(disc_output, torch.zeros_like(disc_output))

In [ ]:
from tqdm.auto import tqdm
from torch.cuda.amp import autocast, GradScaler


# scaler
scaler = GradScaler()

def train_epoch(generator, discriminator, dataloader, optimizer_G, optimizer_D,
                stft_module, mrstft_loss, mag_loss, perceptual_loss, adv_loss,
                device, epoch, log_interval=10):
    generator.train()
    discriminator.train()
    total_loss_G = 0
    total_loss_D = 0

    # loss weights
    m = config["mag_ri_weight"]
    lambda_Ltf = config["lambda_Ltf"]
    lambda_Lgan = config["lambda_Lgan"]
    lambda_Lpercep = config["lambda_Lpercep"]
    lambda_Lmrstft = config["lambda_Lmrstft"]
    
    progress_bar = tqdm(total=len(dataloader), dynamic_ncols=True, leave=False, position=0, desc='Train')
    
    for batch_idx, (noisy, clean, snr) in enumerate(dataloader):
        noisy, clean = noisy.to(device), clean.to(device)
        B = noisy.size(0)
        # STFT
        noisy_stft, noisy_complex = stft_module(noisy)
        clean_stft, clean_complex = stft_module(clean)
        # For generator input: concatenated magnitude, real, imag (shape: B, T, F, 3) -> permute to (B,3,T,F)
        gen_input = noisy_stft.permute(0,3,1,2)  # (B,3,T,F)

        with torch.cuda.amp.autocast():
            # Forward through generator (use ATCEnhanceNetGenerator here, but we'll pass the appropriate model)
            if isinstance(generator, DeCGANGenerator):
                mask, comp = generator(gen_input)
                # Combine mask and comp to produce enhanced complex spectrogram? This is complex. We'll approximate.
                # For simplicity, we'll treat comp as the enhanced real/imag, and use mask to weight magnitude.
                # But this is not exactly the paper. For now, we'll just compute losses on mask and comp.
                # However, the paper's losses use clean magnitude and real/imag. So we'll assume we can compute:
                # enhanced_mag = mask * noisy_stft[...,0]   (noisy magnitude)
                # enhanced_real, enhanced_imag = comp[...,0], comp[...,1]
                # But this requires noisy_stft magnitude. We'll extract from noisy_stft.
                noisy_mag = noisy_stft[...,0]  # (B,T,F)
                enhanced_mag = mask.squeeze(1) * noisy_mag  # (B,T,F)
                enhanced_real = comp[:,0,...]   # (B,T,F)
                enhanced_imag = comp[:,1,...]   # (B,T,F)
            else:  # ATCEnhanceNetGenerator
                comp, _ = generator(gen_input)  # comp: (B,2,T,F)
                enhanced_real = comp[:,0,...]
                enhanced_imag = comp[:,1,...]
                # For magnitude, we can compute from real/imag
                enhanced_mag = torch.sqrt(enhanced_real**2 + enhanced_imag**2 + 1e-8)
        
        # Clean references
        clean_mag = clean_stft[...,0]  # (B,T,F)
        clean_real = clean_stft[...,1]
        clean_imag = clean_stft[...,2]
        
        # Losses
        L_mag = mag_loss(enhanced_mag, clean_mag) # magnitude loss
        L_ri = F.l1_loss(enhanced_real, clean_real) + F.l1_loss(enhanced_imag, clean_imag) # phase aware loss
        L_tf = m * L_mag + (1 - m) * L_ri
        # MRSTFT loss on waveforms
        # Reconstruct enhanced waveform from enhanced real/imag
        # For simplicity, we'll just use the original noisy phase for reconstruction? Not ideal.
        # Better: use the inverse STFT with the enhanced real/imag and the noisy phase? Let's use the inverse function.
        # We need to reconstruct waveform from enhanced magnitude and phase (computed from enhanced real/imag)
        # Actually we have enhanced_real/imag, so we can reconstruct directly.
        # But the stft_module.inverse expects magnitude and phase? Let's implement a helper.
        # We'll compute phase from real/imag and reconstruct.
        phase_enh = torch.atan2(enhanced_imag, enhanced_real + 1e-8)
        mag_enh = torch.sqrt(enhanced_real**2 + enhanced_imag**2 + 1e-8)
        # clamp
        mag_enh = torch.clamp(mag_enh, min=1e-5, max=1e3)
        # Apply power-law inverse
        mag_enh_orig = mag_enh ** (1/POWER_LAW_C)
        spec_enh = mag_enh_orig * torch.exp(1j * phase_enh)
        # Inverse STFT
        spec_enh = spec_enh.permute(0, 2, 1)  # (B, F, T)
        enhanced_wave = torch.istft(spec_enh, n_fft=N_FFT, hop_length=HOP_LENGTH,
                                    win_length=WIN_LENGTH, window=torch.hann_window(WIN_LENGTH).to(device),
                                    length=noisy.size(-1))
        L_mrstft = mrstft_loss(enhanced_wave, clean.squeeze(1))
        # Perceptual loss
        if batch_idx % 5 == 0:
            L_percep = perceptual_loss(clean, enhanced_wave)
        else:
            L_percep = 0
        # Adversarial loss for generator
        # For discriminator, we need magnitude spectrograms
        disc_input_enh = enhanced_mag.unsqueeze(1)  # (B,1,T,F)
        disc_input_clean = clean_mag.unsqueeze(1)
        fake_pred = discriminator(disc_input_enh)
        L_gan = adv_loss(fake_pred, target_real=True)
        # Generator total loss
        L_G = lambda_Ltf * L_tf + lambda_Lgan * L_gan +  lambda_Lmrstft * L_mrstft + lambda_Lpercep * L_percep  # weights from paper + perceptual
        total_loss_G += L_G.item()
        
        # Discriminator training
        real_pred = discriminator(disc_input_clean)
        L_D_real = adv_loss(real_pred, target_real=True)
        fake_pred = discriminator(disc_input_enh.detach())
        L_D_fake = adv_loss(fake_pred, target_real=False)
        L_D = (L_D_real + L_D_fake) / 2
        total_loss_D += L_D.item()
        
        # Backward
        scaler.scale(L_G).backward()
        torch.nn.utils.clip_grad_norm_(generator.parameters(), 5.0) # clip gradient
        scaler.step(optimizer_G)
        scaler.update()
        
        # optimizer_G.zero_grad()
        # L_G.backward()
        # optimizer_G.step()

        

        scaler.scale(L_D).backward()
        torch.nn.utils.clip_grad_norm_(discriminator.parameters(), 5.0) # clip gradient
        scaler.step(optimizer_D)
        scaler.update()
        # optimizer_D.zero_grad()
        # L_D.backward()
        # optimizer_D.step()

        progress_bar.set_postfix({
            "G": f"{L_G.item():.4f}",
            "D": f"{L_D.item():.4f}"
        })
        
        progress_bar.update() # Update tqdm bar
        # if batch_idx % log_interval == 0:
        #     print(f'Epoch {epoch} [{batch_idx}/{len(dataloader)}] G_loss: {L_G.item():.4f} D_loss: {L_D.item():.4f}')
    
    return total_loss_G / len(dataloader), total_loss_D / len(dataloader)

def validate_epoch(generator, dataloader, stft_module, device, compute_metrics=False):
    generator.eval()
    total_metrics = {'pesq': [], 'stoi': [], 'si_snr': []}
    total_loss = 0
    
    m = config["mag_ri_weight"]
    
    progress_bar = tqdm(total=len(dataloader), dynamic_ncols=True, position=0, leave=False, desc='Val')
    
    with torch.no_grad():
        for noisy, clean, _ in dataloader:
            noisy, clean = noisy.to(device), clean.to(device)
            # STFT
            noisy_stft, _ = stft_module(noisy)
            clean_stft, _ = stft_module(clean)
            gen_input = noisy_stft.permute(0,3,1,2)

            
            with torch.inference_mode():
                if isinstance(generator, DeCGANGenerator):
                    mask, comp = generator(gen_input)
                    # Combine to get enhanced waveform (simplified)
                    noisy_mag = noisy_stft[...,0]
                    enhanced_mag = mask.squeeze(1) * noisy_mag
                    enhanced_real = comp[:,0,...]
                    enhanced_imag = comp[:,1,...]
                else:
                    comp, _ = generator(gen_input)
                    enhanced_real = comp[:,0,...]
                    enhanced_imag = comp[:,1,...]
                # Reconstruct waveform
                phase_enh = torch.atan2(enhanced_imag, enhanced_real + 1e-8)
                mag_enh = torch.sqrt(enhanced_real**2 + enhanced_imag**2 + 1e-8)
                # clamp
                mag_enh = torch.clamp(mag_enh, min=1e-5, max=1e3)
                mag_enh_orig = mag_enh ** (1/POWER_LAW_C)
                spec_enh = mag_enh_orig * torch.exp(1j * phase_enh)
                spec_enh = spec_enh.permute(0, 2, 1)  # (B, F, T)
                # clamp enhanced_real and enhanced_imag
                enhanced_real = torch.clamp(enhanced_real, -1e3, 1e3)
                enhanced_imag = torch.clamp(enhanced_imag, -1e3, 1e3)
                enhanced_wave = torch.istft(spec_enh, n_fft=N_FFT, hop_length=HOP_LENGTH,
                                            win_length=WIN_LENGTH, window=torch.hann_window(WIN_LENGTH).to(device),
                                            length=noisy.size(-1))

            # Compute metrics if needed
            if compute_metrics:
                
                # Magnitude + RI loss (cheap)
                clean_mag = clean_stft[...,0]
                clean_real = clean_stft[...,1]
                clean_imag = clean_stft[...,2]
                
                enhanced_mag = torch.sqrt(enhanced_real**2 + enhanced_imag**2 + 1e-8)
                
                L_mag = F.l1_loss(enhanced_mag, clean_mag)
                L_ri = F.l1_loss(enhanced_real, clean_real) + F.l1_loss(enhanced_imag, clean_imag)
                L_tf = m * L_mag + (1 - m) * L_ri
                
                total_loss += L_tf.item()
                
                for i in range(noisy.size(0)):
                    clean_np = clean[i].cpu().numpy()
                    enh_np = enhanced_wave[i].cpu().numpy()
                    # PESQ
                    try:
                        pesq_score = pesq(SAMPLE_RATE, clean_np, enh_np, 'wb')
                        total_metrics['pesq'].append(pesq_score)
                    except:
                        pass
                    # STOI
                    stoi_score = stoi(clean_np, enh_np, SAMPLE_RATE, extended=False)
                    total_metrics['stoi'].append(stoi_score)
                    # SI-SNR
                    noise = enh_np - clean_np
                    si_snr = 10 * np.log10(np.mean(clean_np**2) / (np.mean(noise**2) + 1e-8))
                    total_metrics['si_snr'].append(si_snr)
                if total_metrics['stoi']:
                    progress_bar.set_postfix({
                        "STOI": f"{np.mean(total_metrics['stoi']):.3f}"
                    })
                    
                    progress_bar.update() # Update tqdm bar
                    
    avg_loss = total_loss / len(dataloader)
    avg_metrics = {k: np.mean(v) for k, v in total_metrics.items() if v}
    return avg_loss, avg_metrics

### Wandb

In [ ]:
import datetime
import wandb

# Use wandb? Resume Training?
USE_WANDB = config['wandb']

RESUME_LOGGING = False # Set this to true if you are resuming training from a previous run

# Create your wandb run

run_name = f"{datetime.datetime.now().strftime('%Y-%m-%d_%H-%M')}_{config['Name']}"

# If you are resuming an old run
if USE_WANDB:

    wandb.login(key="<wandb key>") #TODO
    
    if RESUME_LOGGING:
        run = wandb.init(
            id     = "", ### Insert specific run id here if you want to resume a previous run
            resume = "must", ### You need this to resume previous runs
            project = "ATC-EnhanceNet-ablations", ### Project should be created in your wandb
            settings = wandb.Settings(_service_wait=300)
        )


    else:
        run = wandb.init(
            name    = run_name, ### Wandb creates random run names if you skip this field, we recommend you give useful names
            reinit  = True, ### Allows reinitalizing runs when you re-run this cell
            project = "ATC-EnhanceNet-ablations", ### Project should be created in your wandb account
            config  = config ### Wandb Config for your run
        )

        ### Save your model architecture as a string with str(model)
        model_arch  = f"{str(generator)} \n {str(discriminator)}"
        ### Save it in a txt file
        arch_file   = open("model_arch.txt", "w")
        file_write  = arch_file.write(model_arch)
        arch_file.close()

        ### log it in your wandb run with wandb.save()
        wandb.save('model_arch.txt')

## Training loop

In [ ]:
# To free up ram
import gc
gc.collect()

In [ ]:
start_epoch = 0
best_pesq = -float("inf")
best_stoi = -float("inf")
best_si_snr = -float("inf")
num_epochs = config["epochs"]

# Optimizers and Schedulers
optimizer_G = torch.optim.Adam(generator.parameters(), lr=config["optimizer_G_lr"])
optimizer_D = torch.optim.Adam(discriminator.parameters(), lr=config["optimizer_D_lr"])
scheduler_G = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_G, T_max=100)
scheduler_D = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_D, T_max=100)

# Losses
mrstft_loss = MultiResolutionSTFTLoss().to(device)
mag_loss = ComplexMagnitudeLoss()
perceptual_loss = PerceptualLoss().to(device)
adv_loss = AdversarialLoss()


checkpoint_root = os.path.join(os.getcwd(), 'checkpoints')
os.makedirs(checkpoint_root, exist_ok=True)
last_checkpoint_path = f'{checkpoint_root}/last_checkpoint.pt'
best_pesq_checkpoint_path = f'{checkpoint_root}/best_model_pesq.pt'
best_stoi_checkpoint_path = f'{checkpoint_root}/best_model_stoi.pt'
best_si_snr_checkpoint_path = f'{checkpoint_root}/best_model_si_snr.pt'

In [ ]:
# RESUME_TRAINING = True

# if RESUME_TRAINING:
#     checkpoint = torch.load(last_checkpoint_path, weights_only=False)
    
#     generator.load_state_dict(checkpoint['generator_state_dict'])
#     discriminator.load_state_dict(checkpoint['discriminator_state_dict'])
#     optimizer_G.load_state_dict(checkpoint['optimizer_G_state_dict'])
#     optimizer_D.load_state_dict(checkpoint['optimizer_D_state_dict'])
    
#     start_epoch = checkpoint['epoch'] + 1
#     best_pesq = checkpoint.get('best_pesq', -float("inf"))
#     best_stoi = checkpoint.get('best_stoi', -float("inf"))
#     best_si_snr = checkpoint.get('best_si_snr', -float("inf"))
    
# !pwd

In [ ]:
# Training
for epoch in range(start_epoch, num_epochs+1):
    
    print("\nEpoch: {}/{}".format(epoch + 1, config['epochs']))
    
    train_loss_G, train_loss_D = train_epoch(
        generator, discriminator, train_loader,
        optimizer_G, optimizer_D, stft_module,
        mrstft_loss, mag_loss, perceptual_loss, adv_loss,
        device, epoch
    )
    
    val_loss, metrics = validate_epoch(generator, val_loader, stft_module, device, compute_metrics=True)

    curr_lr_G = optimizer_G.param_groups[0]['lr']
    curr_lr_D = optimizer_D.param_groups[0]['lr']
    
    scheduler_G.step()
    scheduler_D.step()
    
    print(f'Train G Loss: {train_loss_G:.4f} | Train D Loss: {train_loss_D:.4f}')
    print(f'Validation metrics: PESQ={metrics.get("pesq", 0):.4f} | STOI={metrics.get("stoi", 0):.4f} | SI-SNR={metrics.get("si_snr", 0):.2f}')

    # Save last checkpoint
    torch.save({
        'epoch': epoch,
        'generator_state_dict': generator.state_dict(),
        'discriminator_state_dict': discriminator.state_dict(),
        'optimizer_G_state_dict': optimizer_G.state_dict(),
        'optimizer_D_state_dict': optimizer_D.state_dict(),
        'best_pesq': best_pesq,
        'best_stoi': best_stoi,
        'best_si_snr': best_si_snr
    }, last_checkpoint_path)

    if config['wandb']:
        wandb.log({
            'train_loss_G': train_loss_G,
            'train_loss_D': train_loss_D,
            'val_loss': val_loss,
            'val_pesq': metrics["pesq"],
            'val_stoi': metrics["stoi"],
            'val_si_snr': metrics["si_snr"],
            'lr_G': curr_lr_G,
            'lr_D': curr_lr_D,
        })
        wandb.save(last_checkpoint_path)
         
    print("Saved last checkpoint")

    # Extract metrics
    pesq_val = metrics.get("pesq", 0)
    stoi_val = metrics.get("stoi", 0)
    sisnr_val = metrics.get("si_snr", 0)

    # Save best PESQ
    if pesq_val > best_pesq:
        best_pesq = pesq_val
        torch.save({
            'epoch': epoch,
            'generator_state_dict': generator.state_dict(),
            'discriminator_state_dict': discriminator.state_dict(),
            'optimizer_G_state_dict': optimizer_G.state_dict(),
            'optimizer_D_state_dict': optimizer_D.state_dict(),
            'best_pesq': best_pesq,
            'best_stoi': best_stoi,
            'best_si_snr': best_si_snr
        }, best_pesq_checkpoint_path)
        if config['wandb']:
            wandb.save(best_pesq_checkpoint_path)
        print("Saved best PESQ model")

    # Save best STOI
    if stoi_val > best_stoi:
        best_stoi = stoi_val
        torch.save({
            'epoch': epoch,
            'generator_state_dict': generator.state_dict(),
            'discriminator_state_dict': discriminator.state_dict(),
            'optimizer_G_state_dict': optimizer_G.state_dict(),
            'optimizer_D_state_dict': optimizer_D.state_dict(),
            'best_pesq': best_pesq,
            'best_stoi': best_stoi,
            'best_si_snr': best_si_snr
        }, best_stoi_checkpoint_path)
        if config['wandb']:
            wandb.save(best_stoi_checkpoint_path)
        print("Saved best STOI model")

    # Save best SI-SNR
    if sisnr_val > best_si_snr:
        best_si_snr = sisnr_val
        torch.save({
            'epoch': epoch,
            'generator_state_dict': generator.state_dict(),
            'discriminator_state_dict': discriminator.state_dict(),
            'optimizer_G_state_dict': optimizer_G.state_dict(),
            'optimizer_D_state_dict': optimizer_D.state_dict(),
            'best_pesq': best_pesq,
            'best_stoi': best_stoi,
            'best_si_snr': best_si_snr
        }, best_si_snr_checkpoint_path)
        if config['wandb']:
            wandb.save(best_si_snr_checkpoint_path)
        print("Saved best SI-SNR model")

## Prediction

Generate enhanced audio file from noisy audio file

In [ ]:
# Load best checkpoint

checkpoint = torch.load(best_pesq_checkpoint_path, weights_only=False)

generator.load_state_dict(checkpoint['generator_state_dict'])

In [ ]:
def predict(generator, stft_module, test_audio_path, output_path, device):
    """Enhance a single audio file and save the result."""
    generator.eval()
    audio, sr = librosa.load(test_audio_path, sr=SAMPLE_RATE, mono=True)
    audio_t = torch.from_numpy(audio).float().unsqueeze(0).to(device)  # (1, T)
    with torch.no_grad():
        noisy_stft, _ = stft_module(audio_t)
        gen_input = noisy_stft.permute(0,3,1,2)
        if isinstance(generator, DeCGANGenerator):
            mask, comp = generator(gen_input)
            noisy_mag = noisy_stft[...,0]
            enhanced_mag = mask.squeeze(1) * noisy_mag
            enhanced_real = comp[:,0,...]
            enhanced_imag = comp[:,1,...]
        else:
            comp, _ = generator(gen_input)
            enhanced_real = comp[:,0,...]
            enhanced_imag = comp[:,1,...]
        phase_enh = torch.atan2(enhanced_imag, enhanced_real + 1e-8)
        mag_enh = torch.sqrt(enhanced_real**2 + enhanced_imag**2 + 1e-8)
        mag_enh_orig = mag_enh ** (1/POWER_LAW_C)
        spec_enh = mag_enh_orig * torch.exp(1j * phase_enh)
        spec_enh = spec_enh.permute(0, 2, 1)  # (B, F, T)
        enhanced_wave = torch.istft(spec_enh, n_fft=N_FFT, hop_length=HOP_LENGTH,
                                    win_length=WIN_LENGTH, window=torch.hann_window(WIN_LENGTH).to(device),
                                    length=audio_t.size(-1))
        enhanced = enhanced_wave.squeeze().cpu().numpy()
    sf.write(output_path, enhanced, SAMPLE_RATE)
    print(f"Enhanced audio saved to {output_path}")

In [ ]:
prediction_root = os.path.join(os.getcwd(), 'predictions')
os.makedirs(prediction_root, exist_ok=True)

audio_files = clean_files[:10] # First 10
for test_audio_path in audio_files:
    file_name = os.path.splitext(os.path.basename(test_audio_path))[0]
    output_path = os.path.join(prediction_root, file_name + ".wav")
    #print(f"{test_audio_path} -> {output_path}")
    predict(generator, stft_module, test_audio_path, output_path, device)


## Evaluation

In [ ]:
def transcribe_wav2vec(audio_segment, processor, model, device):
    """
    Transcribe a single audio segment (numpy array) using the ASR model.
    """
    # Ensure correct shape and dtype
    if audio_segment.dtype != np.float32:
        audio_segment = audio_segment.astype(np.float32)
    # Normalize
    audio_segment = audio_segment / (np.max(np.abs(audio_segment)) + 1e-8)
    # Process
    inputs = processor(audio_segment, sampling_rate=SAMPLE_RATE, return_tensors="pt", padding=True)
    input_values = inputs.input_values.to(device)
    with torch.no_grad():
        logits = model(input_values).logits
        predicted_ids = torch.argmax(logits, dim=-1)
        transcription = processor.decode(predicted_ids[0])
    return transcription.lower().strip()

import xml.etree.ElementTree as ET
from tqdm.auto import tqdm

def evaluate_wer_on_test_set(generator, stft_module, dataloader, device, asr_model, processor):
    generator.eval()
    asr_model.eval()
    
    all_segments = []

    # Wrap dataloader with tqdm
    for batch in tqdm(dataloader, desc="Processing batches", unit="batch"):
        audio_batch, xml_paths = batch
        if isinstance(audio_batch, list):
            audio_batch = torch.nn.utils.rnn.pad_sequence(audio_batch, batch_first=True).to(device)
        else:
            audio_batch = audio_batch.to(device)
        
        batch_size = audio_batch.size(0)
        
        # Optionally, add inner progress bar for segments
        for i in tqdm(range(batch_size), desc="Processing audios", leave=False, unit="audio"):
            audio_t = audio_batch[i].unsqueeze(0)
            xml_path = xml_paths[i]
            audio_np = audio_t.squeeze().cpu().numpy()
            
            # -------- Enhance the audio ----------
            with torch.no_grad():
                noisy_stft, _ = stft_module(audio_t)
                gen_input = noisy_stft.permute(0,3,1,2)
                if isinstance(generator, DeCGANGenerator):
                    mask, comp = generator(gen_input)
                    noisy_mag = noisy_stft[...,0]
                    enhanced_mag = mask.squeeze(1) * noisy_mag
                    enhanced_real = comp[:,0,...]
                    enhanced_imag = comp[:,1,...]
                else:
                    comp, _ = generator(gen_input)
                    enhanced_real = comp[:,0,...]
                    enhanced_imag = comp[:,1,...]

                phase_enh = torch.atan2(enhanced_imag, enhanced_real + 1e-8)
                mag_enh = torch.sqrt(enhanced_real**2 + enhanced_imag**2 + 1e-8)
                mag_enh_orig = mag_enh ** (1/POWER_LAW_C)
                spec_enh = mag_enh_orig * torch.exp(1j * phase_enh)
                spec_enh = spec_enh.permute(0, 2, 1)
                enhanced_wave = torch.istft(
                    spec_enh,
                    n_fft=N_FFT,
                    hop_length=HOP_LENGTH,
                    win_length=WIN_LENGTH,
                    window=torch.hann_window(WIN_LENGTH).to(device),
                    length=audio_t.size(-1)
                )
                enhanced_audio = enhanced_wave.squeeze().cpu().numpy()
            
            # -------- Parse XML segments ----------
            tree = ET.parse(xml_path)
            root = tree.getroot()
            segments = []
            for seg in root.findall('segment'):
                start = float(seg.find('start').text)
                end = float(seg.find('end').text)
                text = seg.find('text').text.strip().lower()
                tags = seg.find('tags')
                if tags is not None:
                    correct_transcript = tags.find('correct_transcript')
                    if correct_transcript is not None and correct_transcript.text == '0':
                        continue
                segments.append((start, end, text))
            
            # -------- Compute WER for each segment ----------
            for start, end, ref_text in segments:
                start_sample = int(start * SAMPLE_RATE)
                end_sample = int(end * SAMPLE_RATE)
                
                noisy_seg = audio_np[start_sample:end_sample]
                enhanced_seg = enhanced_audio[start_sample:end_sample]
                
                hyp_noisy = transcribe_wav2vec(noisy_seg, processor, asr_model, device)
                hyp_enhanced = transcribe_wav2vec(enhanced_seg, processor, asr_model, device)
                
                wer_noisy = wer(ref_text, hyp_noisy)
                wer_enhanced = wer(ref_text, hyp_enhanced)
                
                all_segments.append({
                    'file': os.path.basename(xml_path),
                    'start': start,
                    'end': end,
                    'ref': ref_text,
                    'hyp_noisy': hyp_noisy,
                    'hyp_enhanced': hyp_enhanced,
                    'wer_noisy': wer_noisy,
                    'wer_enhanced': wer_enhanced
                })
    
    df = pd.DataFrame(all_segments)
    avg_wer_noisy = df['wer_noisy'].mean()
    avg_wer_enhanced = df['wer_enhanced'].mean()
    improvement = avg_wer_noisy - avg_wer_enhanced
    
    print(f"Average WER on noisy: {avg_wer_noisy:.4f}")
    print(f"Average WER on enhanced: {avg_wer_enhanced:.4f}")
    print(f"Absolute improvement: {improvement:.4f} ({improvement*100:.2f}%)")
    
    return df, avg_wer_noisy, avg_wer_enhanced, improvement

In [ ]:
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor

# Load ASR model for speech-to-text
asr_model = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-base-960h").to(device)
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")

# Evaluate
df_wer, avg_noisy, avg_enhanced, imp = evaluate_wer_on_test_set(
    generator, stft_module, test_loader, device, asr_model, processor
)